In [47]:
import sys
from matplotlib import pyplot
from keras.datasets import cifar10
from keras.utils import to_categorical
from keras.layers import Conv2D, Flatten, Dense, BatchNormalization, Input, MaxPooling2D
from keras.models import Sequential
from keras.optimizers import SGD, Adam, RMSprop, Adagrad, Adamax, Nadam, Ftrl
from keras.callbacks import EarlyStopping
from hyperopt import hp, tpe, fmin, Trials, space_eval, STATUS_OK

In [48]:
def load_dataset():
  (trainX,trainY),(testX,testY) = cifar10.load_data()
  trainY = to_categorical(trainY)
  testY = to_categorical(testY)
  return trainX, trainY, testX, testY

# normalizando as imagens
def prep_pixels(train,test):
  train_norm = train.astype('float32')
  test_norm = test.astype('float32')
  train_norm = train_norm/255.0
  test_norm = test_norm/255.0
  return train_norm, test_norm

In [49]:
param_space = {
    'lr': hp.loguniform('lr',-9,-3),
    "optimizer": hp.choice("optimizer",[SGD, Adam, RMSprop, Adagrad, Adamax, Nadam, Ftrl]),
    "epochs": hp.uniform("epochs",10,100),
    'batch_size':hp.choice('batch_size',[32,64,128]),
    "hiddenLayerOne": hp.uniform("hiddenLayerOne",10,100),
    "hiddenLayerTwo": hp.uniform("hiddenLayerTwo",10,100),
    "dropout": hp.choice("dropout", [0.1, 0.4, 0.6])
  }

In [50]:
from keras.src.layers import MaxPooling2D, Dropout
## Dropout regularization and BatchNormalization

def define_model(params):
  model = Sequential()
  model.add(Conv2D(32,(3,3), activation='relu', kernel_initializer = 'he_uniform', padding = "same", input_shape = (32, 32, 3) ))
  model.add(Conv2D(32,(3,3), activation='relu', kernel_initializer = 'he_uniform', padding = "same"))
  model.add(BatchNormalization())
  model.add(MaxPooling2D((2,2)))
  model.add(Dropout((0.2), noise_shape = None))
  model.add(Conv2D(64,(3,3), activation='relu', kernel_initializer='he_uniform', padding='same'))
  model.add(Conv2D(64,(3,3), activation='relu', kernel_initializer='he_uniform', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPooling2D((2,2)))
  model.add(Dropout((0.2), noise_shape = None))
  model.add(Conv2D(128, (3,3), activation = 'relu', kernel_initializer= 'he_uniform', padding = "same"))
  model.add(Conv2D(128, (3,3), activation = 'relu', kernel_initializer= 'he_uniform', padding = "same"))
  model.add(BatchNormalization())
  model.add(MaxPooling2D((2,2)))
  model.add(Dropout(params['dropout'], noise_shape = None))
  model.add(Flatten())
  model.add(Dense(params['hiddenLayerOne'], activation=params['activation'], input_dim=124))
  model.add(BatchNormalization())
  model.add(Dropout((0.2), noise_shape = None))
  model.add(Dense(params['hiddenLayerOne'], activation=params['activation'], input_dim=124))
  model.add(layers.Dropout(params['dropout']))
  opt = SGD(learning_rate=0.001, momentum=0.9)
  model.compile(optimizer=params['optimizer'](params['lr']), loss= 'categorical_crossentropy', metrics = ['accuracy'])
  es = EarlyStopping(monitor='val_loss',mode='min',
                       verbose=1,patience=15)
  
  return model


In [51]:
# plot das curvas diagnosticas de aprendizado
def summarize_diagnostics(history):
  ## plot loss
  pyplot.subplot(211)
  pyplot.title('Cross Entropy Loss')
  pyplot.plot(history.history['loss'], color = 'blue', label = 'train')
  pyplot.plot(history.history['val_loss'], color = 'orange', label = 'test')
  ## plot acuracia
  pyplot.subplot(212)
  pyplot.title('Classification Accuracy')
  pyplot.plot(history.history['accuracy'], color = 'blue', label = 'train')
  pyplot.plot(history.history['val_accuracy'], color = 'orange', label = 'test')
  ## save plot to file
  filename = sys.argv[0].split('/')[-1]
  pyplot.savefig(filename + '_plot.png')
  pyplot.close()

In [52]:
def run_test_harness(params):
  trainX, trainY, testX, testY = load_dataset()
  trainX, testX = prep_pixels(trainX, testX)
  model = define_model()
  history = model.fit(trainX, trainY, epochs = params['epochs'], batch_size = int(params['batch_size']), validation_data = (testX, testY), verbose = 0)
  _, acc = model.evaluate(testX, testY, verbose = 0)
  print('> %.3f' % (acc*100.0))
  summarize_diagnostics(history)
  print('Test accuracy:', acc)
  return {'loss': -acc, 
          'status': STATUS_OK, 
          'model': model,
          'params': params}

In [54]:
trials = Trials()
best_params = (
  model=run_test_harness,
  space=param_space,
  algo=tpe.suggest,
  max_evals=200,
  trial=trials)
space_eval(param_space, best_param)

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (159541847.py, line 3)